In [1]:
from google.colab import files

uploaded = files.upload()

Saving rule_engine.py to rule_engine.py
Saving phq_risk_model.pkl to phq_risk_model.pkl


In [2]:
import os

print(os.listdir("/content"))

['.config', 'phq_risk_model.pkl', 'rule_engine.py', 'sample_data']


In [3]:
import joblib

model = joblib.load("/content/phq_risk_model.pkl")

print("Model loaded successfully!")
print("Model type:", type(model))

Model loaded successfully!
Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [4]:
print("Model input features:")
print(model.feature_names_in_)

print("\nModel output classes:")
print(model.classes_)

Model input features:
['Sleep Quality' 'Study Pressure' 'Financial Pressure']

Model output classes:
['High' 'Low' 'Moderate']


In [5]:
import pandas as pd

# Test input
test_data = pd.DataFrame([{
    "Sleep Quality": 2,
    "Study Pressure": 3,
    "Financial Pressure": 2
}])

# Make prediction
prediction = model.predict(test_data)[0]

# Get prediction probabilities
probabilities = model.predict_proba(test_data)[0]

print("Test input:")
print(test_data)

print("\nPredicted lifestyle risk:", prediction)

print("\nPrediction probabilities:")
for class_name, probability in zip(model.classes_, probabilities):
    print(f"{class_name}: {probability:.2%}")

Test input:
   Sleep Quality  Study Pressure  Financial Pressure
0              2               3                   2

Predicted lifestyle risk: High

Prediction probabilities:
High: 67.35%
Low: 0.41%
Moderate: 32.24%


In [6]:
!pip install flask flask-cors

In [7]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import joblib
import sys

# Make sure Colab can find rule_engine.py
sys.path.append("/content")

# Import the existing PHQ-9 rule engine
from rule_engine import calculate_severity

# Load the trained Random Forest model
model = joblib.load("/content/phq_risk_model.pkl")

# Create Flask application
app = Flask(__name__)

# Allow requests from your frontend
CORS(app)

print("Flask imported successfully!")
print("PHQ-9 rule engine imported successfully!")
print("Random Forest model loaded successfully!")

Flask imported successfully!
PHQ-9 rule engine imported successfully!
Random Forest model loaded successfully!


In [8]:
import rule_engine

print("Functions/classes available in rule_engine.py:")
print([name for name in dir(rule_engine) if not name.startswith("_")])

Functions/classes available in rule_engine.py:
['SEVERITY_BANDS', 'calculate_severity']


In [9]:
import inspect
import rule_engine

print("Function signature:")
print(inspect.signature(rule_engine.calculate_severity))

print("\nFunction source:")
print(inspect.getsource(rule_engine.calculate_severity))

Function signature:
(answers: list[int]) -> dict

Function source:
def calculate_severity(answers: list[int]) -> dict:
    """
    answers: list of 9 integers (0-3), one per PHQ-9 question, in order.
    Returns dict with total score, severity label, and a suicide-risk flag.
    """
    if len(answers) != 9 or any(a not in (0, 1, 2, 3) for a in answers):
        raise ValueError("Expected 9 answers, each between 0 and 3.")

    total = sum(answers)
    severity = next(label for low, high, label in SEVERITY_BANDS if low <= total <= high)

    # Question 9 = suicidal ideation item. Flag independently of total score
    # so the recommendation layer can prioritize crisis resources.
    suicide_risk = answers[8] >= 2

    return {
        "phq_total": total,
        "phq_severity": severity,
        "suicide_risk_flag": suicide_risk,
    }



In [10]:
from rule_engine import calculate_severity

# Test PHQ-9 answers
test_answers = [0, 1, 1, 2, 1, 0, 1, 2, 0]

result = calculate_severity(test_answers)

print("PHQ-9 test result:")
print(result)

PHQ-9 test result:
{'phq_total': 8, 'phq_severity': 'Mild', 'suicide_risk_flag': False}


In [11]:
import joblib
from rule_engine import calculate_severity

# Load the trained Random Forest model
model = joblib.load("/content/phq_risk_model.pkl")

# -----------------------------
# Test PHQ-9 answers
# -----------------------------
phq_answers = [0, 1, 1, 2, 1, 0, 1, 2, 0]

phq_result = calculate_severity(phq_answers)

# -----------------------------
# Test lifestyle inputs
# -----------------------------
sleep_quality = 2
study_pressure = 3
financial_pressure = 2

lifestyle_input = [[
    sleep_quality,
    study_pressure,
    financial_pressure
]]

lifestyle_prediction = model.predict(lifestyle_input)[0]
lifestyle_probabilities = model.predict_proba(lifestyle_input)[0]

# -----------------------------
# Display combined result
# -----------------------------
print("===== COMBINED ASSESSMENT TEST =====")

print("\nPHQ-9:")
print("Score:", phq_result["phq_total"])
print("Severity:", phq_result["phq_severity"])
print("Suicide-risk flag:", phq_result["suicide_risk_flag"])

print("\nLifestyle ML:")
print("Risk:", lifestyle_prediction)

print("\nPrediction probabilities:")
for class_name, probability in zip(model.classes_, lifestyle_probabilities):
    print(f"{class_name}: {probability * 100:.2f}%")

===== COMBINED ASSESSMENT TEST =====

PHQ-9:
Score: 8
Severity: Mild
Suicide-risk flag: False

Lifestyle ML:
Risk: High

Prediction probabilities:
High: 67.35%
Low: 0.41%
Moderate: 32.24%


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [12]:
import pandas as pd
import joblib

# Load the trained model
model = joblib.load("/content/phq_risk_model.pkl")

# Use the exact feature names the model was trained with
lifestyle_input = pd.DataFrame([{
    "Sleep Quality": 2,
    "Study Pressure": 3,
    "Financial Pressure": 2
}])

# Predict lifestyle risk
lifestyle_prediction = model.predict(lifestyle_input)[0]

# Get prediction probabilities
lifestyle_probabilities = model.predict_proba(lifestyle_input)[0]

print("Lifestyle risk:", lifestyle_prediction)

print("\nPrediction probabilities:")
for class_name, probability in zip(model.classes_, lifestyle_probabilities):
    print(f"{class_name}: {probability * 100:.2f}%")

Lifestyle risk: High

Prediction probabilities:
High: 67.35%
Low: 0.41%
Moderate: 32.24%


In [13]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import pandas as pd
import joblib

from rule_engine import calculate_severity


# --------------------------------------------------
# 1. Create Flask application
# --------------------------------------------------

app = Flask(__name__)

# Allow your frontend to communicate with Flask
CORS(app)


# --------------------------------------------------
# 2. Load trained Random Forest model
# --------------------------------------------------

model = joblib.load("/content/phq_risk_model.pkl")


# --------------------------------------------------
# 3. Prediction API
# --------------------------------------------------

@app.route("/predict", methods=["POST"])
def predict():

    try:

        # Get JSON data sent by frontend
        data = request.get_json()

        # ------------------------------------------
        # PHQ-9 answers
        # ------------------------------------------

        answers = data.get("answers")

        # ------------------------------------------
        # Lifestyle variables
        # ------------------------------------------

        sleep_quality = data.get("sleep_quality")
        study_pressure = data.get("study_pressure")
        financial_pressure = data.get("financial_pressure")


        # ------------------------------------------
        # Validate PHQ-9 answers
        # ------------------------------------------

        if not isinstance(answers, list):
            return jsonify({
                "success": False,
                "error": "PHQ-9 answers must be provided as a list."
            }), 400

        if len(answers) != 9:
            return jsonify({
                "success": False,
                "error": "Exactly 9 PHQ-9 answers are required."
            }), 400


        # ------------------------------------------
        # Calculate official PHQ-9 result
        # ------------------------------------------

        phq_result = calculate_severity(answers)


        # ------------------------------------------
        # Prepare lifestyle data
        # ------------------------------------------

        lifestyle_input = pd.DataFrame([{
            "Sleep Quality": sleep_quality,
            "Study Pressure": study_pressure,
            "Financial Pressure": financial_pressure
        }])


        # ------------------------------------------
        # Random Forest prediction
        # ------------------------------------------

        lifestyle_prediction = model.predict(
            lifestyle_input
        )[0]


        lifestyle_probabilities = model.predict_proba(
            lifestyle_input
        )[0]


        probability_dict = {
            class_name: round(float(probability), 4)
            for class_name, probability
            in zip(model.classes_, lifestyle_probabilities)
        }


        # ------------------------------------------
        # Final response
        # ------------------------------------------

        return jsonify({

            "success": True,

            "phq9": {
                "score": phq_result["phq_total"],
                "severity": phq_result["phq_severity"],
                "suicide_risk_flag": phq_result["suicide_risk_flag"]
            },

            "lifestyle_risk": lifestyle_prediction,

            "lifestyle_probabilities": probability_dict

        })


    except ValueError as e:

        return jsonify({
            "success": False,
            "error": str(e)
        }), 400


    except Exception as e:

        return jsonify({
            "success": False,
            "error": "An unexpected server error occurred.",
            "details": str(e)
        }), 500


# --------------------------------------------------
# 4. Test route
# --------------------------------------------------

@app.route("/", methods=["GET"])
def home():

    return jsonify({
        "status": "online",
        "message": "Mental Health Assessment API is running."
    })


print("Flask API created successfully!")
print("Available endpoints:")
print("GET  /")
print("POST /predict")

Flask API created successfully!
Available endpoints:
GET  /
POST /predict


In [14]:
!pip install pyngrok

In [15]:
from pyngrok import ngrok

# Close any previous tunnels
ngrok.kill()

# Start Flask in the background
from threading import Thread

def run_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

flask_thread = Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

print("Flask server started on port 5000")

Flask server started on port 5000


In [17]:
from pyngrok import ngrok

ngrok.set_auth_token("3I3FCoH9si9zk2a7LoW2LHowStd_845JDcDTyLK7RtVjcwsve")

print("ngrok authentication configured.")

ngrok authentication configured.


In [18]:
from pyngrok import ngrok

# Create a public tunnel to Flask
public_url = ngrok.connect(5000)

print("Your Flask API URL is:")
print(public_url)

Your Flask API URL is:
NgrokTunnel: "https://eastcoast-bullfight-left.ngrok-free.dev" -> "http://localhost:5000"


In [19]:
import requests

# Your public Flask API URL
API_URL = "https://eastcoast-bullfight-left.ngrok-free.dev/predict"

# Test assessment data
test_data = {
    "answers": [1, 1, 1, 1, 1, 1, 1, 0, 0],
    "sleep_quality": 2,
    "study_pressure": 3,
    "financial_pressure": 2
}

# Send data to Flask API
response = requests.post(
    API_URL,
    json=test_data
)

print("HTTP Status:", response.status_code)
print("API Response:")
print(response.json())

INFO:werkzeug:127.0.0.1 - - [22/Aug/2026 12:54:10] "POST /predict HTTP/1.1" 200 -


HTTP Status: 200
API Response:
{'lifestyle_probabilities': {'High': 0.6735, 'Low': 0.0041, 'Moderate': 0.3224}, 'lifestyle_risk': 'High', 'phq9': {'score': 7, 'severity': 'Mild', 'suicide_risk_flag': False}, 'success': True}
